<a href="https://colab.research.google.com/github/yhnx/EN4554_Assignment_2/blob/main/Assignment2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EN4554 – Assignment 2: CNNs (CIFAR-10, PyTorch)

**Name:** `Asuramuni S. Y.`  **Index No.:** `220046R`  
**GitHub:** `https://github.com/yhnx/EN4554_Assignment_2`

**AI-tool acknowledgement:** *I used Claude's help in drafting some code snippets of this notebook to assist with the implementation. I have read, run and understood all of the code, and the interpretations are my own.*



## 0. Setup, data and shared utilities

In [1]:
import os, math, time, random, copy
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision
from torch.optim import Optimizer
from IPython.display import display


SEED = 42


def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.backends.cudnn.benchmark = True
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})
print('device:', device, '| torch', torch.__version__)

device: cuda | torch 2.11.0+cu130


In [ ]:
# ---- Load CIFAR-10, split 45k/5k, normalise with TRAIN statistics ----
def load_cifar10(root='./data'):
    tr = torchvision.datasets.CIFAR10(root, train=True,  download=True)
    te = torchvision.datasets.CIFAR10(root, train=False, download=True)
    f = lambda a: torch.from_numpy(a).permute(0, 3, 1, 2).contiguous()      # uint8, N x 3 x 32 x 32
    return f(tr.data), torch.tensor(tr.targets), f(te.data), torch.tensor(te.targets)

CLASSES = ['plane','car','bird','cat','deer','dog','frog','horse','ship','truck']
X_all, Y_all, X_te, Y_te = load_cifar10()

g = torch.Generator().manual_seed(SEED)
perm = torch.randperm(len(X_all), generator=g)
tr_idx, va_idx = perm[:45000], perm[45000:]

x01 = X_all[tr_idx].float() / 255
MEAN = x01.mean((0, 2, 3), keepdim=True)          # statistics from the training split only
STD  = x01.std((0, 2, 3), keepdim=True)
norm = lambda x: (x.float() / 255 - MEAN) / STD

Xtr, Ytr = norm(X_all[tr_idx]).to(device), Y_all[tr_idx].to(device)
Xva, Yva = norm(X_all[va_idx]).to(device), Y_all[va_idx].to(device)
Xte, Yte = norm(X_te), Y_te                        # kept on CPU, NOT used until Task 4
del x01
print('train', tuple(Xtr.shape), '| val', tuple(Xva.shape), '| mean', MEAN.flatten().tolist(), '| std', STD.flatten().tolist())

fig, axs = plt.subplots(2, 8, figsize=(12, 3.4))
for a, i in zip(axs.flat, range(16)):
    img = (Xtr[i].cpu() * STD[0] + MEAN[0]).clamp(0, 1).permute(1, 2, 0)
    a.imshow(img); a.set_title(CLASSES[Ytr[i].item()], fontsize=8); a.axis('off')
plt.tight_layout(); plt.show()

  0%|          | 295k/170M [00:07<1:09:08, 41.0kB/s]

In [ ]:
# ---- Shared utilities: evaluation, GPU augmentations (cutout / mixup written by hand), training loop ----
def amp_ctx(on):
    return torch.autocast(device_type=device, dtype=torch.float16, enabled=(on and device == 'cuda'))

def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

@torch.no_grad()
def evaluate(model, X, Y, bs=1000, amp=False):
    model.eval(); loss, correct = 0., 0
    for i in range(0, len(X), bs):
        with amp_ctx(amp):
            out = model(X[i:i+bs])
        loss += F.cross_entropy(out.float(), Y[i:i+bs], reduction='sum').item()
        correct += (out.argmax(1) == Y[i:i+bs]).sum().item()
    return loss / len(X), correct / len(X)

def random_flip_crop(x, pad=4):
    """Random horizontal flip + random 32x32 crop from the zero-padded image (per-sample, vectorised)."""
    N, C, H, W = x.shape; d = x.device
    flip = torch.rand(N, device=d) < 0.5
    x = torch.where(flip[:, None, None, None], x.flip(3), x)
    xp = F.pad(x, (pad, pad, pad, pad))
    oy = torch.randint(0, 2 * pad + 1, (N,), device=d); ox = torch.randint(0, 2 * pad + 1, (N,), device=d)
    ar = torch.arange(H, device=d)
    rows = (oy[:, None] + ar)[:, None, :, None]
    cols = (ox[:, None] + ar)[:, None, None, :]
    n = torch.arange(N, device=d)[:, None, None, None]; c = torch.arange(C, device=d)[None, :, None, None]
    return xp[n, c, rows, cols]

def cutout(x, size=16):
    """Cutout: zero (= the mean colour after normalisation) a size x size square at a random centre; may be clipped at borders."""
    N, C, H, W = x.shape; d = x.device
    cy = torch.randint(0, H, (N, 1), device=d); cx = torch.randint(0, W, (N, 1), device=d)
    ar = torch.arange(H, device=d)[None]
    my = (ar >= cy - size // 2) & (ar < cy + size // 2)
    mx = (ar >= cx - size // 2) & (ar < cx + size // 2)
    mask = (my[:, :, None] & mx[:, None, :])[:, None]
    return x.masked_fill(mask, 0.)

def mixup_batch(x, y, alpha=0.4):
    """Mixup: x~ = lam*x_i + (1-lam)*x_j, lam ~ Beta(alpha, alpha). Loss = lam*CE(y_i) + (1-lam)*CE(y_j)."""
    lam = float(np.random.beta(alpha, alpha))
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def warmup_cosine(opt, total_steps, warmup_steps):
    def f(s):
        if s < warmup_steps: return (s + 1) / warmup_steps
        p = (s - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1 + math.cos(math.pi * p))
    return torch.optim.lr_scheduler.LambdaLR(opt, f)

def train(model, opt, X, Y, Xv, Yv, epochs, bs=128, flipcrop=False, cut=False, mix=False,
          sched=None, label_smoothing=0.0, amp=False, eval_train=False, verbose=True, name=''):
    """Generic training loop. Returns history dict. If eval_train, train accuracy is measured on the clean
    (un-augmented) training set in eval mode; otherwise it is the running accuracy during the epoch."""
    scaler = torch.amp.GradScaler(device, enabled=(amp and device == 'cuda'))
    hist = {k: [] for k in ['train_loss', 'train_acc', 'val_loss', 'val_acc']}; hist['diverged'] = False
    nb = max(1, len(X) // bs)
    for e in range(epochs):
        model.train(); t0 = time.time()
        perm = torch.randperm(len(X), device=X.device)
        run_loss = torch.zeros((), device=X.device); run_corr = torch.zeros((), device=X.device); seen = 0
        for b in range(nb):
            idx = perm[b * bs:(b + 1) * bs]
            xb, yb = X[idx], Y[idx]
            if flipcrop: xb = random_flip_crop(xb)
            if cut:      xb = cutout(xb)
            with amp_ctx(amp):
                if mix:
                    xb, ya, yb2, lam = mixup_batch(xb, yb)
                    out = model(xb)
                    loss = lam * F.cross_entropy(out, ya, label_smoothing=label_smoothing) \
                         + (1 - lam) * F.cross_entropy(out, yb2, label_smoothing=label_smoothing)
                    tgt = ya
                else:
                    out = model(xb); loss = F.cross_entropy(out, yb, label_smoothing=label_smoothing); tgt = yb
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            if sched is not None: sched.step()
            run_loss += loss.detach() * len(idx); run_corr += (out.argmax(1) == tgt).sum(); seen += len(idx)
        tl = (run_loss / seen).item()
        if not math.isfinite(tl):                       # training blew up
            hist['diverged'] = True
            hist['train_loss'].append(float('nan')); hist['train_acc'].append(0.1)
            hist['val_loss'].append(float('nan')); hist['val_acc'].append(0.1)
            if verbose: print(f'{name} epoch {e+1}: DIVERGED');
            break
        vl, va = evaluate(model, Xv, Yv, amp=amp)
        ta = evaluate(model, X, Y, amp=amp)[1] if eval_train else (run_corr / seen).item()
        for k, v in zip(['train_loss', 'train_acc', 'val_loss', 'val_acc'], [tl, ta, vl, va]): hist[k].append(v)
        if verbose and (e % max(1, epochs // 10) == 0 or e == epochs - 1):
            print(f'{name} ep {e+1:3d}/{epochs} | train loss {tl:.3f} acc {ta:.3f} | val loss {vl:.3f} acc {va:.3f} | {time.time()-t0:.1f}s')
    return hist

---
# Task 1: Backpropagation

## 1(a) Layers with explicit forward / backward (no autograd)
For a minibatch of size $N$: `Linear`: $Y = XW + b$, so $\partial L/\partial W = X^\top G$, $\partial L/\partial b = \sum_n G_n$, $\partial L/\partial X = G W^\top$.
`ReLU`: $G \odot \mathbb 1[X>0]$. Softmax cross-entropy: $\partial L/\partial z = (\mathrm{softmax}(z) - \mathrm{onehot}(y))/N$.
A `Sigmoid` layer is also provided for Task 1(d) (derivative $\sigma(1-\sigma)$).

In [ ]:
class Linear:
    def __init__(self, din, dout, init='he', dtype=torch.float32, device='cpu'):
        std = {'he': math.sqrt(2 / din), 'xavier': math.sqrt(2 / (din + dout)), 'small': 0.01}[init]
        self.W = torch.randn(din, dout, dtype=dtype, device=device) * std
        self.b = torch.zeros(dout, dtype=dtype, device=device)
    def forward(self, x):
        self.x = x
        return x @ self.W + self.b
    def backward(self, g):
        self.dW = self.x.T @ g
        self.db = g.sum(0)
        return g @ self.W.T

class ReLU:
    def forward(self, x):
        self.mask = x > 0
        return x * self.mask
    def backward(self, g):
        return g * self.mask

class Sigmoid:
    def forward(self, x):
        self.y = torch.sigmoid(x)
        return self.y
    def backward(self, g):
        return g * self.y * (1 - self.y)

class SoftmaxCE:
    def forward(self, logits, y):
        z = logits - logits.max(1, keepdim=True).values            # numerical stability
        logp = z - torch.logsumexp(z, 1, keepdim=True)
        self.p, self.y = logp.exp(), y
        return -logp[torch.arange(len(y)), y].mean()
    def backward(self):
        g = self.p.clone()
        g[torch.arange(len(self.y)), self.y] -= 1
        return g / len(self.y)

class MLP:
    def __init__(self, sizes, act='relu', init='he', dtype=torch.float32, device='cpu'):
        self.layers = []
        for i in range(len(sizes) - 1):
            self.layers.append(Linear(sizes[i], sizes[i + 1], init, dtype, device))
            if i < len(sizes) - 2:
                self.layers.append(ReLU() if act == 'relu' else Sigmoid())
        self.loss_fn = SoftmaxCE()
    def forward(self, x):
        for l in self.layers: x = l.forward(x)
        return x
    def loss(self, x, y):
        return self.loss_fn.forward(self.forward(x), y)
    def backward(self):
        g = self.loss_fn.backward()
        for l in reversed(self.layers): g = l.backward(g)
    def linears(self):
        return [l for l in self.layers if isinstance(l, Linear)]

## 1(b) Gradient check: autograd and centred finite differences
Float64 network `3072-64-32-10` on 32 real images. Relative error $=\|a-b\|/(\|a\|+\|b\|)$ per parameter tensor.
Finite differences, $\frac{L(\theta+\epsilon)-L(\theta-\epsilon)}{2\epsilon}$ with $\epsilon=10^{-6}$, are evaluated on 10 randomly chosen entries of every tensor (all entries would be too slow).
`torch.allclose` is used as the pass/fail criterion.

In [ ]:
def autograd_grads(mlp, x, y):
    """Same network re-implemented with autograd (copies of the same weights)."""
    P = []
    for L in mlp.linears():
        P += [L.W.clone().requires_grad_(True), L.b.clone().requires_grad_(True)]
    h, n = x, len(mlp.linears())
    for i in range(n):
        h = h @ P[2 * i] + P[2 * i + 1]
        if i < n - 1: h = torch.relu(h)
    loss = F.cross_entropy(h, y); loss.backward()
    return loss.item(), [p.grad for p in P]

relerr = lambda a, b: ((a - b).norm() / (a.norm() + b.norm() + 1e-300)).item()

set_seed()
chk = MLP([3072, 64, 32, 10], act='relu', init='he', dtype=torch.float64, device='cpu')
xc, yc = Xtr[:32].flatten(1).double().cpu(), Ytr[:32].cpu()
loss_mine = chk.loss(xc, yc).item(); chk.backward()
loss_ag, g_ag = autograd_grads(chk, xc, yc)
print(f'loss (mine) {loss_mine:.10f} | loss (autograd) {loss_ag:.10f}')

names  = [f'{k}{i+1}' for i in range(len(chk.linears())) for k in ('W', 'b')]
g_mine = [t for L in chk.linears() for t in (L.dW, L.db)]
params = [t for L in chk.linears() for t in (L.W, L.b)]
EPS, rows = 1e-6, []
for nm, gm, ga, P in zip(names, g_mine, g_ag, params):
    sel = [tuple(int(i) for i in np.unravel_index(int(k), P.shape)) for k in torch.randperm(P.numel())[:10]]
    fd, an = [], []
    for ix in sel:
        old = P[ix].item()
        P[ix] = old + EPS; lp = chk.loss(xc, yc).item()
        P[ix] = old - EPS; lm = chk.loss(xc, yc).item()
        P[ix] = old
        fd.append((lp - lm) / (2 * EPS)); an.append(gm[ix].item())
    fd, an = torch.tensor(fd, dtype=torch.float64), torch.tensor(an, dtype=torch.float64)
    rows.append({'param': nm, 'rel.err vs autograd': relerr(gm, ga),
                 'allclose vs autograd': torch.allclose(gm, ga, rtol=1e-6, atol=1e-10),
                 'rel.err vs finite-diff': relerr(an, fd),
                 'allclose vs finite-diff': torch.allclose(an, fd, rtol=1e-4, atol=1e-7)})
display(pd.DataFrame(rows).style.format({'rel.err vs autograd': '{:.2e}', 'rel.err vs finite-diff': '{:.2e}'}))

**Interpretation (write in your own words):** *e.g.* report the order of magnitude of the errors against autograd (expected $\sim10^{-15}$–$10^{-12}$, i.e. float64 round-off) and against finite differences (expected $\sim10^{-8}$–$10^{-6}$, limited by the truncation error $O(\epsilon^2)$ and round-off $O(u/\epsilon)$). Mention that both checks pass, so the backward formulas are right, and why finite differences are only a *check* (cost $2$ forward passes **per parameter**) while backprop gives all gradients for the cost of ~1 extra forward pass.

## 1(c) Train the MLP with plain SGD (our own backprop)
`3072-512-256-10`, He init, minibatch 128, lr 0.05, 20 epochs, on flattened normalised images.

In [ ]:
Xtr_f, Xva_f = Xtr.flatten(1), Xva.flatten(1)
set_seed()
mlp = MLP([3072, 512, 256, 10], act='relu', init='he', device=device)
EPOCHS_1C, LR_1C, BS_1C = 20, 0.05, 128
it_loss, va_hist, tr_hist = [], [], []
for e in range(EPOCHS_1C):
    perm = torch.randperm(len(Xtr_f), device=device)
    for b in range(len(perm) // BS_1C):
        idx = perm[b * BS_1C:(b + 1) * BS_1C]
        it_loss.append(mlp.loss(Xtr_f[idx], Ytr[idx]))
        mlp.backward()
        for L in mlp.linears():                       # plain SGD step
            L.W -= LR_1C * L.dW; L.b -= LR_1C * L.db
    acc = lambda X, Y: (mlp.forward(X).argmax(1) == Y).float().mean().item()
    va_hist.append(acc(Xva_f, Yva)); tr_hist.append(acc(Xtr_f, Ytr))
    print(f'epoch {e+1:2d} | train acc {tr_hist[-1]:.3f} | val acc {va_hist[-1]:.3f}')
it_loss = torch.stack(it_loss).cpu().numpy()

w = min(100, len(it_loss))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(it_loss, alpha=0.3, label='per-iteration'); ax[0].plot(np.arange(w - 1, len(it_loss)), np.convolve(it_loss, np.ones(w) / w, 'valid'), label=f'moving avg ({w})')
ax[0].set(xlabel='iteration', ylabel='training loss', title='MLP trained with own backprop + SGD'); ax[0].legend()
ax[1].plot(range(1, EPOCHS_1C + 1), tr_hist, label='train'); ax[1].plot(range(1, EPOCHS_1C + 1), va_hist, label='val')
ax[1].set(xlabel='epoch', ylabel='accuracy', title='Accuracy'); ax[1].legend()
plt.tight_layout(); plt.show()
print(f'Final validation accuracy: {va_hist[-1]:.4f}')

**Interpretation (write in your own words):** describe the loss curve (noisy per-iteration, smooth trend), the final validation accuracy (an MLP on raw pixels typically reaches ~50–57 %), and the growing train–val gap (an MLP has no spatial inductive bias and overfits; motivates CNNs).

## 1(d) Per-layer gradient norms at initialisation (10-layer MLP)
Width 256, 10 `Linear` layers, one batch of 256 images, gradient norm of each weight matrix (layer 1 = closest to the input). Log scale.
Sigmoid is paired with Xavier init (its natural choice), ReLU with He init; ReLU with a tiny $N(0,0.01^2)$ init is added to show that the initialisation matters as well.

In [ ]:
def layer_grad_norms(act, init, depth=10, width=256, n=256):
    set_seed()
    m = MLP([3072] + [width] * (depth - 1) + [10], act=act, init=init, device=device)
    m.loss(Xtr[:n].flatten(1), Ytr[:n]); m.backward()
    return [L.dW.norm().item() for L in m.linears()]

cfgs = {'sigmoid + Xavier': ('sigmoid', 'xavier'), 'ReLU + He': ('relu', 'he'), 'ReLU + N(0, 0.01²)': ('relu', 'small')}
norms = {k: layer_grad_norms(*v) for k, v in cfgs.items()}
plt.figure(figsize=(6.5, 4))
for k, v in norms.items(): plt.semilogy(range(1, 11), v, 'o-', label=k)
plt.xlabel('layer (1 = input side)'); plt.ylabel('||dL/dW|| (log)'); plt.title('Gradient norm per layer at initialisation'); plt.legend(); plt.show()
display(pd.DataFrame(norms, index=[f'layer {i}' for i in range(1, 11)]).style.format('{:.2e}'))

**Explanation (write in your own words; hints):** the gradient reaching layer $l$ is a product of $W^\top$ and activation-derivative factors over all layers above it. The sigmoid derivative is $\le 0.25$, so each layer multiplies the signal by a factor well below 1 and the gradient **vanishes** towards the input (layer 1 ≪ layer 10) – the norms differ by several orders of magnitude. ReLU has derivative 1 on its active half, and He init ($\mathrm{Var}(W)=2/n_{in}$) preserves the variance through ReLU layers, so the norms stay at a similar scale across depth. With a too-small init even ReLU vanishes (each layer shrinks by ≈ $0.01\sqrt{n_{in}/2}$).

---
# Task 2: Optimizers

## 2(a) SGD with momentum and Adam as `torch.optim.Optimizer` subclasses
Conventions follow PyTorch: momentum buffer $v \leftarrow \mu v + g$, $\theta \leftarrow \theta - \eta v$ (first step $v=g$);
Adam: $m\leftarrow\beta_1 m+(1-\beta_1)g$, $v\leftarrow\beta_2 v+(1-\beta_2)g^2$, $\hat m=m/(1-\beta_1^t)$, $\hat v=v/(1-\beta_2^t)$, $\theta\leftarrow\theta-\eta\,\hat m/(\sqrt{\hat v}+\epsilon)$. Weight decay is the L2 form ($g \leftarrow g + \lambda\theta$), as in `torch.optim`.

In [ ]:
class MySGD(Optimizer):
    def __init__(self, params, lr=0.01, momentum=0.0, weight_decay=0.0):
        super().__init__(params, dict(lr=lr, momentum=momentum, weight_decay=weight_decay))
    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad(): loss = closure()
        for grp in self.param_groups:
            for p in grp['params']:
                if p.grad is None: continue
                d = p.grad
                if grp['weight_decay'] != 0: d = d + grp['weight_decay'] * p
                if grp['momentum'] != 0:
                    st = self.state[p]
                    if 'buf' not in st: st['buf'] = d.clone()
                    else: st['buf'].mul_(grp['momentum']).add_(d)
                    d = st['buf']
                p.add_(d, alpha=-grp['lr'])
        return loss

class MyAdam(Optimizer):
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0):
        super().__init__(params, dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay))
    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad(): loss = closure()
        for grp in self.param_groups:
            b1, b2 = grp['betas']
            for p in grp['params']:
                if p.grad is None: continue
                d = p.grad
                if grp['weight_decay'] != 0: d = d + grp['weight_decay'] * p
                st = self.state[p]
                if not st:
                    st['t'] = 0; st['m'] = torch.zeros_like(p); st['v'] = torch.zeros_like(p)
                st['t'] += 1; t = st['t']
                st['m'].mul_(b1).add_(d, alpha=1 - b1)
                st['v'].mul_(b2).addcmul_(d, d, value=1 - b2)
                m_hat = st['m'] / (1 - b1 ** t)                      # bias correction
                v_hat = st['v'] / (1 - b2 ** t)
                p.addcdiv_(m_hat, v_hat.sqrt().add_(grp['eps']), value=-grp['lr'])
        return loss

In [ ]:
def compare(make_mine, make_ref, steps=100):
    set_seed(1)
    net1 = nn.Sequential(nn.Linear(20, 32), nn.Tanh(), nn.Linear(32, 3)).double()
    net2 = copy.deepcopy(net1)
    o1, o2 = make_mine(net1.parameters()), make_ref(net2.parameters())
    x, y = torch.randn(64, 20, dtype=torch.float64), torch.randint(0, 3, (64,))
    for _ in range(steps):
        for n, o in ((net1, o1), (net2, o2)):
            o.zero_grad(); F.cross_entropy(n(x), y).backward(); o.step()
    diff = max((p - q).abs().max().item() for p, q in zip(net1.parameters(), net2.parameters()))
    return diff, all(torch.allclose(p, q, atol=1e-10) for p, q in zip(net1.parameters(), net2.parameters()))

tests = {
    'SGD (lr .1)':                          (lambda p: MySGD(p, lr=.1),                              lambda p: torch.optim.SGD(p, lr=.1)),
    'SGD momentum .9, wd 1e-3':             (lambda p: MySGD(p, lr=.05, momentum=.9, weight_decay=1e-3), lambda p: torch.optim.SGD(p, lr=.05, momentum=.9, weight_decay=1e-3)),
    'Adam (lr 1e-2)':                       (lambda p: MyAdam(p, lr=1e-2),                           lambda p: torch.optim.Adam(p, lr=1e-2)),
    'Adam (lr 1e-2, wd 1e-3, betas .8/.9)': (lambda p: MyAdam(p, lr=1e-2, betas=(.8, .9), weight_decay=1e-3), lambda p: torch.optim.Adam(p, lr=1e-2, betas=(.8, .9), weight_decay=1e-3)),
}
rows = [{'case': k, 'max |param diff| after 100 steps': (r := compare(*v))[0], 'allclose': r[1]} for k, v in tests.items()]
display(pd.DataFrame(rows).style.format({'max |param diff| after 100 steps': '{:.2e}'}))

**Interpretation:** state that the parameters agree to round-off level (float64, ~1e-15) over 100 steps for every setting, i.e. the implementations are equivalent to `torch.optim.SGD` / `torch.optim.Adam`.

## 2(b) Trajectories on the ill-conditioned quadratic $f(x,y)=\tfrac12(x^2+50y^2)$
Hessian eigenvalues 1 and 50 (condition number 50). Start at $(8,1)$, 100 steps. Learning rates: SGD 0.035 (stability limit is $2/50=0.04$), momentum 0.9 with 0.01, Adam 0.3.

In [ ]:
def run_quadratic(make_opt, start=(8., 1.), steps=100):
    w = torch.tensor(start, requires_grad=True); opt = make_opt([w])
    path, fs = [w.detach().numpy().copy()], []
    for _ in range(steps):
        opt.zero_grad(); f = 0.5 * (w[0] ** 2 + 50 * w[1] ** 2); f.backward(); fs.append(f.item()); opt.step()
        path.append(w.detach().numpy().copy())
    return np.array(path), fs

runs = {'SGD (lr 0.035)':                 run_quadratic(lambda p: MySGD(p, lr=0.035)),
        'SGD + momentum (lr 0.01, μ 0.9)': run_quadratic(lambda p: MySGD(p, lr=0.01, momentum=0.9)),
        'Adam (lr 0.3)':                  run_quadratic(lambda p: MyAdam(p, lr=0.3))}

xs, ys = np.linspace(-9, 9, 400), np.linspace(-1.7, 1.7, 400); XX, YY = np.meshgrid(xs, ys)
ZZ = 0.5 * (XX ** 2 + 50 * YY ** 2)
fig, ax = plt.subplots(1, 4, figsize=(17, 3.8), gridspec_kw={'width_ratios': [1, 1, 1, 1.1]})
for a, (k, (path, fs)) in zip(ax[:3], runs.items()):
    a.contour(XX, YY, ZZ, levels=np.geomspace(0.1, 100, 12), cmap='viridis', alpha=0.7)
    a.plot(path[:, 0], path[:, 1], '.-', ms=4, lw=1, color='C3'); a.plot(*path[0], 'ko'); a.plot(0, 0, 'k*', ms=10)
    a.set(title=k, xlabel='x', ylabel='y', xlim=(-9, 9), ylim=(-1.7, 1.7))
for k, (path, fs) in runs.items(): ax[3].semilogy(fs, label=k)
ax[3].set(xlabel='iteration', ylabel='f(x, y)', title='Objective value'); ax[3].legend(fontsize=7)
plt.tight_layout(); plt.show()

**Explanation (write in your own words; hints):** SGD must use a step below $2/\lambda_{max}=0.04$ for the steep $y$ direction, which makes it zig-zag in $y$ while crawling in the flat $x$ direction (rate $1-\eta\lambda_{min}=0.965$ per step). Momentum averages the oscillating $y$-gradients (they cancel) and accumulates the consistent $x$-gradients (effective step ≈ $\eta/(1-\mu)$), so it moves faster along $x$ but overshoots/spirals a little. Adam normalises each coordinate by its own gradient scale, so both coordinates move at a comparable speed (≈ lr per step), giving a nearly straight path – it is not hurt by the condition number, but with a fixed lr it oscillates near the optimum at the scale of the lr.

## 2(c) Optimizer sensitivity on a small CNN
Small CNN (3 conv blocks 32-64-128 + linear, **no batch norm** so that optimizer sensitivity is not hidden), 10 epochs, batch 128, no augmentation, three learning rates per optimizer (decade apart). Our own `MySGD` / `MyAdam` are used. A diverged run is shown as chance accuracy (0.10).

In [ ]:
def small_cnn(bn=False, p_drop=0.0):
    def blk(i, o):
        L = [nn.Conv2d(i, o, 3, padding=1, bias=not bn)]
        if bn: L.append(nn.BatchNorm2d(o))
        return L + [nn.ReLU(inplace=True), nn.MaxPool2d(2)]
    return nn.Sequential(*blk(3, 32), *blk(32, 64), *blk(64, 128), nn.Flatten(), nn.Dropout(p_drop), nn.Linear(128 * 16, 10))

GRID = {'SGD':          [3e-3, 3e-2, 3e-1],
        'SGD+momentum': [1e-3, 1e-2, 1e-1],      # momentum 0.9 -> effective lr ~10x larger
        'Adam':         [1e-4, 1e-3, 1e-2]}
def make_opt(name, params, lr):
    return {'SGD': lambda: MySGD(params, lr=lr), 'SGD+momentum': lambda: MySGD(params, lr=lr, momentum=0.9),
            'Adam': lambda: MyAdam(params, lr=lr)}[name]()

res2 = {}
for name, lrs in GRID.items():
    for lr in lrs:
        set_seed(); m = small_cnn().to(device)
        res2[(name, lr)] = train(m, make_opt(name, m.parameters(), lr), Xtr, Ytr, Xva, Yva, ep(10), name=f'{name} lr={lr:g}', verbose=False)
        h = res2[(name, lr)]; print(f'{name:13s} lr={lr:<7g} final val acc {h["val_acc"][-1]:.4f}  final train loss {h["train_loss"][-1]:.4f}' + ('  (diverged)' if h['diverged'] else ''))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
best = {}
for i, (name, lrs) in enumerate(GRID.items()):
    accs = [res2[(name, lr)]['val_acc'][-1] for lr in lrs]
    ax[0].semilogx(lrs, accs, 'o-', label=name, color=f'C{i}')
    best[name] = lrs[int(np.argmax(accs))]
    ax[1].plot(range(1, len(res2[(name, best[name])]['train_loss']) + 1), res2[(name, best[name])]['train_loss'], label=f'{name} (lr={best[name]:g})', color=f'C{i}')
ax[0].set(xlabel='learning rate', ylabel='final validation accuracy', title='Sensitivity to learning rate'); ax[0].legend()
ax[1].set(xlabel='epoch', ylabel='training loss', title='Training loss at the best learning rate'); ax[1].legend()
plt.tight_layout(); plt.show()
display(pd.DataFrame([{'optimizer': k[0], 'lr': k[1], 'val acc': v['val_acc'][-1], 'train loss': v['train_loss'][-1], 'diverged': v['diverged']} for k, v in res2.items()]).style.format({'val acc': '{:.4f}', 'train loss': '{:.4f}', 'lr': '{:g}'}))

**Discussion (write in your own words; hints):** which optimizer has the widest range of good learning rates (usually Adam is least sensitive, plain SGD most), which lr diverged or under-trained (too small → slow, too large → divergence), how the best-lr training loss curves compare (convergence speed vs. final loss), and how momentum shifts the optimal lr down by ≈ $1/(1-\mu)$.

---
# Task 3: Regularization and residual connections

## 3(a) Regularisation on 5 000 training images
Small CNN **with batch norm**, trained on the first 5 000 training images, validated on the full 5k validation split. Same recipe for every row (SGD+Nesterov, lr 0.05, warm-up + cosine, batch 128, 60 epochs); only the regulariser changes.
Weight decay 5e-4; dropout p=0.5 before the classifier; flip + 4-px-pad random crop; cutout 16×16; mixup α=0.4 (cutout and mixup implemented above). Train accuracy is measured on the clean training subset in eval mode so that it is comparable across rows.

In [ ]:
X5, Y5 = Xtr[:5000], Ytr[:5000]
CFG3A = {'baseline': {}, 'weight decay': dict(wd=5e-4), 'dropout': dict(p_drop=0.5), 'flip+crop': dict(flipcrop=True),
         'cutout': dict(cut=True), 'mixup': dict(mix=True),
         'all combined': dict(wd=5e-4, p_drop=0.5, flipcrop=True, cut=True, mix=True)}
EP3A = 60
res3a = {}
for name, c in CFG3A.items():
    set_seed(); model = small_cnn(bn=True, p_drop=c.get('p_drop', 0.0)).to(device)
    opt = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9, nesterov=True, weight_decay=c.get('wd', 0.0))
    spe = max(1, len(X5) // 128); sched = warmup_cosine(opt, EP3A * spe, spe)
    res3a[name] = train(model, opt, X5, Y5, Xva, Yva, EP3A, flipcrop=c.get('flipcrop', False), cut=c.get('cut', False),
                        mix=c.get('mix', False), sched=sched, eval_train=True, name=name, verbose=False)
    h = res3a[name]; print(f'{name:13s} train {h["train_acc"][-1]:.4f}  val {h["val_acc"][-1]:.4f}')

rows = [{'method': k, 'train acc': h['train_acc'][-1], 'val acc': h['val_acc'][-1], 'gap (train-val)': h['train_acc'][-1] - h['val_acc'][-1],
         'best val acc': max(h['val_acc'])} for k, h in res3a.items()]
display(pd.DataFrame(rows).style.format({c: '{:.4f}' for c in ['train acc', 'val acc', 'gap (train-val)', 'best val acc']}))

In [ ]:
fig, axs = plt.subplots(2, 4, figsize=(17, 6.5), sharey=True)
for a, (k, h) in zip(axs.flat, res3a.items()):
    a.plot(h['train_acc'], label='train (clean)'); a.plot(h['val_acc'], label='val'); a.set_title(k); a.set_xlabel('epoch')
axs.flat[0].set_ylabel('accuracy'); axs.flat[0].legend()
for k, h in res3a.items(): axs.flat[7].plot(h['val_acc'], label=k)
axs.flat[7].set_title('validation accuracy, all'); axs.flat[7].set_xlabel('epoch'); axs.flat[7].legend(fontsize=6)
plt.tight_layout(); plt.show()

**Interpretation (write in your own words; hints):** the baseline reaches ~100 % train accuracy with a large train–val gap (memorisation of 5k images). Compare which single regulariser closes the gap / improves validation accuracy most (data-driven ones – flip+crop, cutout, mixup – usually help more than weight decay or dropout here, because BN already makes the effect of weight decay indirect), whether regularisers *lower train accuracy* (the gap shrinks), whether mixup slows convergence, and whether combining all is best or over-regularises a small network.

## 3(b) Plain vs. residual CIFAR networks of depth $6n+2$ (8, 20, 56)
Three stages (16/32/64 channels, $n$ basic blocks each: conv-BN-ReLU-conv-BN), global average pooling, linear classifier, Kaiming-normal init. The two families share **exactly the same code and parameters**; the only difference is the identity shortcut $y=\mathrm{ReLU}(F(x)+x)$ (at stage transitions the shortcut is parameter-free: stride-2 subsampling + zero-padded channels). Training: SGD+Nesterov lr 0.1, wd 5e-4, 1-epoch warm-up + cosine, flip+crop, fp16 autocast, 45k training images.
The plotted training loss is the running loss in train mode (with augmentation) – identical protocol for both families.

In [ ]:
class BasicBlock(nn.Module):
    def __init__(self, cin, cout, stride, residual):
        super().__init__()
        self.conv1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False); self.bn1 = nn.BatchNorm2d(cout)
        self.conv2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False);     self.bn2 = nn.BatchNorm2d(cout)
        self.residual, self.down, self.padc = residual, (stride != 1 or cin != cout), (cout - cin) // 2
    def forward(self, x):
        out = self.bn2(self.conv2(F.relu(self.bn1(self.conv1(x)))))
        if self.residual:                                    # the ONLY difference between plain and residual nets
            sc = x
            if self.down: sc = F.pad(x[:, :, ::2, ::2], (0, 0, 0, 0, self.padc, self.padc))
            out = out + sc
        return F.relu(out)

class CifarNet(nn.Module):
    def __init__(self, n, widths=(16, 32, 64), residual=True, num_classes=10):
        super().__init__()
        self.conv1, self.bn1 = nn.Conv2d(3, widths[0], 3, 1, 1, bias=False), nn.BatchNorm2d(widths[0])
        blocks, cin = [], widths[0]
        for i, w in enumerate(widths):
            for j in range(n):
                blocks.append(BasicBlock(cin, w, 2 if (i > 0 and j == 0) else 1, residual)); cin = w
        self.blocks, self.fc = nn.Sequential(*blocks), nn.Linear(cin, num_classes)
        for m in self.modules():
            if isinstance(m, nn.Conv2d): nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d): nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear): nn.init.kaiming_normal_(m.weight); nn.init.zeros_(m.bias)
    def forward(self, x):
        x = self.blocks(F.relu(self.bn1(self.conv1(x))))
        return self.fc(F.adaptive_avg_pool2d(x, 1).flatten(1))

for d in (8, 20, 56):
    n = (d - 2) // 6
    a, b = CifarNet(n, residual=False), CifarNet(n, residual=True)
    assert count_params(a) == count_params(b)
    print(f'depth {d} (n={n}): {count_params(a):,} params (plain and residual identical)')

In [ ]:
EP3B, res3b = ep(30), {}
for d in (8, 20, 56):
    for residual in (False, True):
        key = f"{'ResNet' if residual else 'Plain'}-{d}"
        set_seed(); model = CifarNet((d - 2) // 6, residual=residual).to(device)
        opt = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, nesterov=True, weight_decay=5e-4)
        spe = len(Xtr) // 128; sched = warmup_cosine(opt, EP3B * spe, spe)
        t0 = time.time()
        res3b[key] = train(model, opt, Xtr, Ytr, Xva, Yva, EP3B, flipcrop=True, sched=sched, amp=True, name=key, verbose=True)
        print(f'--> {key}: {time.time()-t0:.0f}s, final train loss {res3b[key]["train_loss"][-1]:.4f}, val acc {res3b[key]["val_acc"][-1]:.4f}\n')

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4))
col = {8: 'C0', 20: 'C1', 56: 'C3'}
for k, h in res3b.items():
    d = int(k.split('-')[1]); ls = '-' if k.startswith('ResNet') else '--'
    ax[0].plot(range(1, len(h['train_loss']) + 1), h['train_loss'], ls, color=col[d], label=k)
    ax[1].plot(range(1, len(h['train_loss']) + 1), h['train_loss'], ls, color=col[d], label=k)
    ax[2].plot(range(1, len(h['val_acc']) + 1), h['val_acc'], ls, color=col[d], label=k)
ax[0].set(xlabel='epoch', ylabel='training loss', title='Training loss (solid = residual, dashed = plain)'); ax[0].legend(fontsize=7)
ax[1].set(xlabel='epoch', ylabel='training loss (log)', title='Same, log scale', yscale='log')
ax[2].set(xlabel='epoch', ylabel='validation accuracy', title='Validation accuracy'); ax[2].legend(fontsize=7)
plt.tight_layout(); plt.show()
display(pd.DataFrame([{'model': k, 'depth': int(k.split('-')[1]), 'final train loss': h['train_loss'][-1], 'min train loss': min(h['train_loss']),
                       'final val acc': h['val_acc'][-1], 'diverged': h['diverged']} for k, h in res3b.items()]).style.format({'final train loss': '{:.4f}', 'min train loss': '{:.4f}', 'final val acc': '{:.4f}'}))

**Explanation (write in your own words; hints):** this is the *degradation problem* (He et al., 2016): the plain-56 reaches a **higher training loss** than plain-20 although it contains it as a sub-family – so it is an optimisation problem, not overfitting. Residual nets show the opposite ordering: deeper = lower training loss, since $y=x+F(x)$ lets the block start near identity (easy to fit when extra depth is not needed) and gives gradients a direct additive path to earlier layers (no vanishing product, cf. Task 1(d)). Note how close/different the depth-8 and depth-20 pairs are, and comment on the validation accuracy.

---
# Task 4 (optional): Best CNN with ≤ 500k parameters and ≤ 20 epochs

**Design:** CIFAR ResNet (basic blocks, parameter-free shortcuts, global average pooling) with 3 stages of widths 20/40/80 and $n=3$ blocks (depth 20), ≈ 0.43 M parameters. Recipe: SGD+Nesterov (momentum 0.9), peak lr tuned on the validation split, 1-epoch warm-up + cosine decay, weight decay 5e-4, flip+crop + cutout, label smoothing 0.1, fp16 autocast, batch 128, **20 epochs**.
Tuning (lr) and the ablation use the validation split only. The test set is used once at the very end.

In [ ]:
BASE = dict(n=3, widths=(20, 40, 80), residual=True, opt='sgd', lr=0.1, wd=5e-4, flipcrop=True, cut=True, ls=0.1)
EP4 = ep(20)

def run_cfg(cfg, tag):
    set_seed(); model = CifarNet(cfg['n'], cfg['widths'], cfg['residual']).to(device)
    npar = count_params(model); assert npar <= 500_000, npar
    if cfg['opt'] == 'sgd': opt = torch.optim.SGD(model.parameters(), lr=cfg['lr'], momentum=0.9, nesterov=True, weight_decay=cfg['wd'])
    else:                   opt = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['wd'])
    spe = len(Xtr) // 128; sched = warmup_cosine(opt, EP4 * spe, spe)
    h = train(model, opt, Xtr, Ytr, Xva, Yva, EP4, flipcrop=cfg['flipcrop'], cut=cfg['cut'], label_smoothing=cfg['ls'],
              sched=sched, amp=True, name=tag, verbose=False)
    print(f'{tag:34s} params {npar:,} | final val acc {h["val_acc"][-1]:.4f} | best {max(h["val_acc"]):.4f}')
    return dict(cfg=cfg, hist=h, params=npar, state={k: v.detach().cpu().clone() for k, v in model.state_dict().items()})

# ---- 1) tune the peak learning rate on the validation split ----
sweep = {lr: run_cfg({**BASE, 'lr': lr}, f'lr={lr}') for lr in (0.05, 0.1, 0.2)}
best_lr = max(sweep, key=lambda lr: sweep[lr]['hist']['val_acc'][-1]); print('best lr:', best_lr)
FINAL = {**BASE, 'lr': best_lr}

In [ ]:
# ---- 2) ablation: remove / change ONE design choice at a time (validation accuracy) ----
ABL = {'full model (reference)':          {},
       'no shortcut (plain net)':         dict(residual=False),
       'no augmentation':                 dict(flipcrop=False, cut=False),
       'no cutout':                       dict(cut=False),
       'no weight decay':                 dict(wd=0.0),
       'no label smoothing':              dict(ls=0.0),
       'AdamW instead of SGD (lr 3e-3)':  dict(opt='adamw', lr=3e-3, wd=5e-2)}
abl = {'full model (reference)': sweep[best_lr]}                      # identical config + seed -> reuse the run
for name, ch in ABL.items():
    if name not in abl: abl[name] = run_cfg({**FINAL, **ch}, name)
ref = abl['full model (reference)']['hist']['val_acc'][-1]
df = pd.DataFrame([{'variant': k, 'params': v['params'], 'final val acc': v['hist']['val_acc'][-1], 'best val acc': max(v['hist']['val_acc']),
                    'Δ vs full': v['hist']['val_acc'][-1] - ref} for k, v in abl.items()])
display(df.style.format({'final val acc': '{:.4f}', 'best val acc': '{:.4f}', 'Δ vs full': '{:+.4f}', 'params': '{:,}'}))
plt.figure(figsize=(8, 3.6)); plt.barh(df['variant'][::-1], df['final val acc'][::-1], color='C0'); plt.xlim(max(0, df['final val acc'].min() - 0.05), 1)
plt.xlabel('validation accuracy'); plt.title('Ablation (20 epochs each)'); plt.tight_layout(); plt.show()

**Ablation discussion (write in your own words):** rank the design choices by how much validation accuracy they cost when removed (typically augmentation and the shortcut matter most for a 20-epoch budget; weight decay / label smoothing smaller effects) and comment on SGD vs AdamW. Note differences < ~0.3 % are within seed noise (single seed per run) – say so.

In [ ]:
# ---- 3) FINAL: evaluate the chosen model on the test set exactly ONCE ----
assert 'TEST_DONE' not in globals(), 'The test set has already been used once - do not re-evaluate!'
final = abl['full model (reference)']
net = CifarNet(FINAL['n'], FINAL['widths'], FINAL['residual']).to(device); net.load_state_dict(final['state'])
test_loss, test_acc = evaluate(net, Xte.to(device), Yte.to(device), amp=True); TEST_DONE = True
print(f'Final model: {final["params"]:,} parameters, {EP4} epochs | val acc {final["hist"]["val_acc"][-1]:.4f} | TEST ACCURACY {test_acc:.4f}')
h = final['hist']
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(h['train_loss'], label='train (aug)'); ax[0].plot(h['val_loss'], label='val'); ax[0].set(xlabel='epoch', title='Loss'); ax[0].legend()
ax[1].plot(h['val_acc'], label='val acc'); ax[1].set(xlabel='epoch', title=f'Validation accuracy (test acc = {test_acc:.4f})'); ax[1].legend()
plt.tight_layout(); plt.show()

**Summary (write in your own words):** final test accuracy, parameter count, epochs; what mattered most; what you would try next with more budget.

---
*Reminder before export:* replace every “write in your own words” block with your own interpretation of **your** plots/tables (they carry marks), fill in name/index/GitHub link at the top, keep the AI-tool acknowledgement accurate, then `File → Print → Save as PDF` as `<index>_a02.pdf`.